# Sentinel-1 and Sentinel-2 scenes used per tile

**Scalable burned area mapping via Sentinel-1/2 fusion** — Tarazona, Y. and Mantas, V. (Appendix A)

This notebook lists the Sentinel-1 and Sentinel-2 scenes used to build the pre- and post-fire composites of each training (TRN1–TRN6) and test (TST1–TST4) tile. It applies the same selection criteria described in the manuscript:

- **Sentinel-2** (`COPERNICUS/S2_SR_HARMONIZED`): scene-level cloud cover below 90%.
- **Sentinel-1** (`COPERNICUS/S1_GRD`): IW mode, VV and VH polarizations, 10 m, ascending and descending orbits combined.
- **Both sensors**: only scenes covering at least 5% of the tile are kept.
- **Temporal windows**: as reported in Table 2 of the manuscript.

The notebook produces two tables:
1. **Number of scenes per tile and window**, which reproduces Table 1 of the manuscript.
2. **List of scene identifiers**, with acquisition date, sensor, window, orbit pass and overlap with the tile.

## 1. Setup and authentication

In [ ]:
# Install the Earth Engine API if needed (e.g., in Google Colab)
# !pip install earthengine-api pandas

import ee
import pandas as pd

# Authenticate and initialize Earth Engine.
# Replace 'your-gee-project' with your own Google Cloud project ID.
ee.Authenticate()
ee.Initialize(project='your-gee-project')

## 2. Parameters, regions and tiles

In [ ]:
# Sentinel-2 scene-level cloud cover threshold (%), applied to all regions
CLOUD_PRE = 90     # pre-fire window
CLOUD_POST = 90    # post-fire window
MIN_OVERLAP = 0.05 # minimum fraction of the tile covered by a scene

# s1_pass: Sentinel-1 orbit pass: 'BOTH', 'ASCENDING' or 'DESCENDING'
# Dates: [start, end), end date exclusive as in GEE filterDate
regions = [
    {
        'region': 'Portugal',
        'tiles': 'projects/ee-geoyons/assets/tiles_trn_tst_port',
        'names': {1: 'TST1', 2: 'TRN3', 3: 'TRN1', 4: 'TRN2'},
        's1_pass': 'BOTH',
        's2': {'pre': ['2017-05-15', '2017-06-15'], 'post': ['2017-10-15', '2017-12-15']},
        's1': {'pre': ['2016-10-15', '2016-12-15'], 'post': ['2017-10-15', '2017-12-15']},
    },
    {
        'region': 'Spain',
        'tiles': 'projects/ee-geoyons/assets/tiles_tst_spain',
        'names': {1: 'TST2'},
        's1_pass': 'BOTH',
        's2': {'pre': ['2020-09-01', '2020-11-30'], 'post': ['2021-09-01', '2021-11-30']},
        's1': {'pre': ['2020-09-01', '2020-11-30'], 'post': ['2021-09-01', '2021-11-30']},
    },
    {
        'region': 'California',
        'tiles': 'projects/ee-geoyons/assets/tiles_trn_tst_cal',
        'names': {1: 'TST3', 2: 'TRN4', 3: 'TRN5', 4: 'TRN6'},
        's1_pass': 'BOTH',
        's2': {'pre': ['2019-10-15', '2019-12-15'], 'post': ['2020-10-15', '2020-12-15']},
        's1': {'pre': ['2019-10-15', '2019-12-15'], 'post': ['2020-10-15', '2020-12-15']},
    },
    {
        'region': 'Canada',
        'tiles': 'projects/ee-geoyons/assets/tiles_tst_can',
        'names': {1: 'TST4'},
        's1_pass': 'BOTH',
        's2': {'pre': ['2020-09-01', '2020-10-30'], 'post': ['2021-09-01', '2021-10-30']},
        's1': {'pre': ['2021-04-01', '2021-05-30'], 'post': ['2021-10-08', '2021-10-17']},
    },
]

## 3. Helper functions

In [ ]:
def filter_overlap(coll, tile_geom):
    # Keep only scenes covering at least MIN_OVERLAP of the tile
    tile_area = tile_geom.area(1)

    def add_overlap(img):
        inter = img.geometry().intersection(tile_geom, ee.ErrorMargin(1)).area(1)
        return img.set('overlapFraction', inter.divide(tile_area))

    return coll.map(add_overlap).filter(ee.Filter.gte('overlapFraction', MIN_OVERLAP))


def s2_scenes(tile_geom, dates, cloud_max):
    # Sentinel-2 surface reflectance scenes
    coll = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(tile_geom)
            .filterDate(dates[0], dates[1])
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_max)))
    return filter_overlap(coll, tile_geom)


def s1_scenes(tile_geom, dates, orbit_pass):
    # Sentinel-1 GRD scenes (IW, VV+VH, 10 m)
    coll = (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(tile_geom)
            .filterDate(dates[0], dates[1])
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
            .filter(ee.Filter.eq('resolution_meters', 10)))
    if orbit_pass != 'BOTH':
        coll = coll.filter(ee.Filter.eq('orbitProperties_pass', orbit_pass))
    return filter_overlap(coll, tile_geom)


def count_pass(coll, orbit_pass):
    # Number of scenes for one orbit pass
    return coll.filter(ee.Filter.eq('orbitProperties_pass', orbit_pass)).size()


def to_rows(coll, info, sensor, window):
    # Convert a scene collection into table rows
    is_s1 = sensor == 'Sentinel-1'

    def make_row(img):
        return ee.Feature(None, {
            'region': info['region'],
            'tile': info['tile'],
            'split': info['split'],
            'sensor': sensor,
            'window': window,
            'scene_id': img.get('system:index'),
            'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
            'orbit_pass': img.get('orbitProperties_pass') if is_s1 else 'NA',
            'overlap_fraction': ee.Number(img.get('overlapFraction')).format('%.3f'),
        })

    return ee.FeatureCollection(coll.map(make_row))

## 4. Select the scenes for each tile and window

In [ ]:
all_rows = ee.FeatureCollection([])
summary = []

for r in regions:
    for tile_id, tile_name in r['names'].items():
        tile_geom = (ee.FeatureCollection(r['tiles'])
                     .filter(ee.Filter.eq('id', tile_id))
                     .geometry())
        info = {
            'region': r['region'],
            'tile': tile_name,
            'split': 'test' if tile_name.startswith('TST') else 'training',
        }

        s2_pre = s2_scenes(tile_geom, r['s2']['pre'], CLOUD_PRE)
        s2_post = s2_scenes(tile_geom, r['s2']['post'], CLOUD_POST)
        s1_pre = s1_scenes(tile_geom, r['s1']['pre'], r['s1_pass'])
        s1_post = s1_scenes(tile_geom, r['s1']['post'], r['s1_pass'])

        all_rows = (all_rows
                    .merge(to_rows(s2_pre, info, 'Sentinel-2', 'pre'))
                    .merge(to_rows(s2_post, info, 'Sentinel-2', 'post'))
                    .merge(to_rows(s1_pre, info, 'Sentinel-1', 'pre'))
                    .merge(to_rows(s1_post, info, 'Sentinel-1', 'post')))

        summary.append(ee.Feature(None, {
            'region': r['region'],
            'tile': tile_name,
            'split': info['split'],
            'optical_pre': s2_pre.size(),
            'optical_post': s2_post.size(),
            'radar_pre': s1_pre.size(),
            'radar_post': s1_post.size(),
            'radar_pre_asc': count_pass(s1_pre, 'ASCENDING'),
            'radar_post_asc': count_pass(s1_post, 'ASCENDING'),
            'radar_pre_dsc': count_pass(s1_pre, 'DESCENDING'),
            'radar_post_dsc': count_pass(s1_post, 'DESCENDING'),
        }))

summary_fc = ee.FeatureCollection(summary)

## 5. Number of scenes per tile and window (Table 1)

In [ ]:
count_cols = ['region', 'tile', 'split', 'optical_pre', 'optical_post',
              'radar_pre', 'radar_post', 'radar_pre_asc', 'radar_post_asc',
              'radar_pre_dsc', 'radar_post_dsc']

df_counts = pd.DataFrame([f['properties'] for f in summary_fc.getInfo()['features']])
df_counts = df_counts[count_cols]
df_counts

## 6. List of scene identifiers

In [ ]:
id_cols = ['region', 'tile', 'split', 'sensor', 'window', 'scene_id',
           'date', 'orbit_pass', 'overlap_fraction']

df_ids = pd.DataFrame([f['properties'] for f in all_rows.getInfo()['features']])
df_ids = df_ids[id_cols].sort_values(['region', 'tile', 'sensor', 'window', 'date'])
print(f'Total number of scenes: {len(df_ids)}')
df_ids.head(20)

## 7. Save the tables

In [ ]:
# Save locally as CSV
df_counts.to_csv('scene_counts_per_tile.csv', index=False)
df_ids.to_csv('scene_ids_per_tile.csv', index=False)

# Optional: export the same tables to Google Drive with Earth Engine
# ee.batch.Export.table.toDrive(collection=summary_fc,
#     description='scene_counts_per_tile', fileFormat='CSV',
#     selectors=count_cols).start()
# ee.batch.Export.table.toDrive(collection=all_rows,
#     description='scene_ids_per_tile', fileFormat='CSV',
#     selectors=id_cols).start()

## 8. Pre-computed tables

The tables generated with this notebook can also be downloaded directly:

- 📊 **Number of scenes per tile and window** (Table 1 of the manuscript): [scene_counts_per_tile.csv](https://drive.google.com/file/d/1GeWQjkYvUTSPexZ5d4uMYRJ4xT0YGGul/view?usp=sharing)
- 🛰️ **Scene identifiers** (scene ID, acquisition date, sensor, window, orbit pass and overlap with the tile, for each tile): [scene_ids_per_tile.csv](https://drive.google.com/file/d/1ksJYDbHQFCK38AsUYM9WTielXoxUr3Ce/view?usp=sharing)